### Setup and Import Libraries

In [3]:
#Import required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier

# Supress warnings for cleaner output
import warnings
warnings.filterwarnings("ignore")


ModuleNotFoundError: No module named 'matplotlib'

### Load and Inspect Data

In [ ]:
#Load the data
data = pd.read_csv("fraud_detection_credit_card.csv")


In [ ]:
data.head(3)

#### Transaction Details:
- trans_date_trans_time: Timestamp of the transaction.
- cc_num: Credit card number (anonymized).
- merchant: Merchant name where the transaction occurred.
- category: Merchant category.
- amt: Transaction amount.
- trans_num: Unique transaction identifier.
- unix_time: UNIX timestamp of the transaction.

#### Customer Information:
- first and last: Customer's first and last name (anonymized).
- gender: Gender of the customer.
- street, city, state, zip: Address information.
- lat and long: Latitude and longitude of the customer's location.
- city_pop: Population of the customer’s city.
- job: Customer's profession.
- dob: Date of birth.
- Customer_Age: Age derived from the dob.

#### Merchant Details:
- merch_lat and merch_long: Latitude and longitude of the merchant’s location.
- merch_zipcode: ZIP code of the merchant (some data missing).

#### Fraud Detection:
- is_fraud: Indicator if the transaction was fraudulent (1 for fraud, 0 for legitimate).

#### Enriched Features:
- Merchant_Category: Mock merchant categories (e.g., Groceries, Electronics).
- Transaction_Type: Online or in-store transactions.
- Customer_Satisfaction_Score: Score from 1 to 10 representing customer satisfaction.
- Transaction_Time: Transaction time in HH:MM format.
- Payment_Method: Payment method (e.g., Credit Card, Debit Card, Mobile Payment).
- Loyalty_Points_Earned: Loyalty points earned in the transaction.

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
# Checking for missing values
data.isnull().sum()

### Preprocessing

In [ ]:
#Drop unnessary or unamed columns
data = data.drop(columns=["Unnamed: 0"])

In [ ]:
data.head(3)

### Visualization

In [ ]:
# Attribute trends
sns.histplot(data["Customer_Satisfaction_Score"],kde=True)
plt.title("Customer Satisfaction Distribution")
plt.show()

In [ ]:
#Relationships (Heatmap and Pairplot)
sns.heatmap(data[["amt","Customer_Age",'Loyalty_Points_Earned']].corr(),annot=True)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
sns.pairplot(data[["amt","Customer_Age",'Loyalty_Points_Earned']])
plt.show()

In [ ]:
#Distribution of Transaction Amounts
plt.Figure(figsize=(8,5))
sns.histplot(data['amt'], bins=100, kde=True,color='blue')
plt.title('Transaction Amount Distribution')
plt.xlabel('Transaction Amount')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Fraud vs Non-Fraud Transaction
plt.figure(figsize=(8,5))
sns.countplot(x='is_fraud',data=data,palette='Set1')
plt.title('Fraudulant vs Non-Fraudulant Transanction')
plt.xlabel('Is Fraud')
plt.ylabel('Count')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x='is_fraud',y='amt', data=data)
plt.title('Transanction Amounts for Fraudulant and Non-Fraudulant Transactions')
plt.show()

In [ ]:
# Transaction Amount by Merchant_Category
plt.figure(figsize=(8,5))
sns.boxplot(x='Merchant_Category',y='amt', data=data)
plt.title('Transanction Amounts by Merchant Category')
plt.show()


In [ ]:
# Transaction Amount by Payment_Method
plt.figure(figsize=(8,5))
sns.boxplot(x='Payment_Method',y='amt', data=data)
plt.title('Transanction Amounts by Payment Method')
plt.show()


### Clustering

In [ ]:
# Preprocessing
data_cluster = data[['amt', 'Customer_Age', 'Loyalty_Points_Earned']].dropna()
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data_cluster)

In [ ]:
# K-Means clustering
inertia = []

for k in range(1,10):
    model = KMeans(n_clusters=k, random_state=42)
    model.fit(data_scaled)
    inertia.append(model.inertia_)

In [ ]:
# Elbow method
plt.plot(range(1,10), inertia, marker="o")
plt.xlabel("Number of Cluster")
plt.ylabel("inertia")
plt.title("Elbow Method")
plt.show()

In [ ]:
#Apply K-means
kmeans = KMeans(n_clusters=5, random_state=42)
kmeans.fit(data_scaled)
labels = kmeans.labels_

In [ ]:
#Visualize clusters with PCA
pca = PCA(n_components=2)
reduced_data = pca.fit_transform(data_scaled)
sns.scatterplot(x=reduced_data[:,0], y=reduced_data[:,1], hue= labels, palette='Set1')
plt.title("Cluster Visualization (PCA)")
plt.show()

### Regression

In [ ]:
# Preprocessing
x = data[['Customer_Age', 'Loyalty_Points_Earned']]
y = data['amt']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

In [ ]:
#Linear Regression
lr = LinearRegression()
lr.fit(x_train,y_train)
y_pred = lr.predict(x_test)

In [ ]:
#Evaluate
print(f"MSE: {mean_squared_error(y_test, y_pred)}")
print(f"R2 Score: {r2_score(y_test, y_pred)}")

### Classification

In [ ]:
# Encoding categorical features using Labelencoder

le = LabelEncoder()
data['Payment_Method_encoded'] = le.fit_transform(data['Payment_Method'].astype(str))
data['Merchant_Category_encoded'] = le.fit_transform(data['Merchant_Category'].astype(str))

In [ ]:
x = data[['Customer_Age','amt','Payment_Method_encoded', 'Merchant_Category_encoded']]
y = data['is_fraud']

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

In [ ]:
# Random Forest Classifier
rfc = RandomForestClassifier()
rfc.fit(x_train, y_train)

In [ ]:
#Evaluate
y_pred = rfc.predict(x_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
import pickle

#Save the trained model
with open ("model.pkl", "wb") as f:
    pickle.dump(rfc, f)

with open("le_payment.pkl", "wb") as f:
    pickle.dump(data['Payment_Method_encoded'], f)

with open("le_merchant.pkl", "wb") as f:
    pickle.dump(data['Merchant_Category_encoded'], f)

print("Model Saved")